# Clustering Comparison — K-Means vs Hierarchical (Ward) vs Mean Shift

> **Dependência:** Corre primeiro `eda.ipynb`, `Kmeans.ipynb`, `Hierarchical.ipynb` e `MeanShift.ipynb`  
> para garantir que os CSVs de assignments existem na pasta.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.cluster import KMeans, AgglomerativeClustering, MeanShift, estimate_bandwidth
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

## 1. Load Data & Re-fit All Models

In [ ]:
costumer_preprocessed = pd.read_csv("costumer_preprocessed_combined.csv")
costumer_basket = pd.read_csv("customer_basket.csv")
costumer_raw = pd.read_csv("customer_info.csv")

basket_agg = (
    costumer_basket
    .groupby('customer_id')
    .agg(total_transactions=('invoice_id', 'count'))
    .reset_index()
)
costumer = pd.merge(costumer_raw, basket_agg, on='customer_id', how='inner').reset_index(drop=True)

print(f"Preprocessed shape: {costumer_preprocessed.shape}")

In [ ]:
CHOSEN_K = 6  # align with the k chosen in Kmeans.ipynb and Hierarchical.ipynb

# --- K-Means ---
kmeans = KMeans(n_clusters=CHOSEN_K, random_state=42, n_init='auto')
labels_kmeans = kmeans.fit_predict(costumer_preprocessed)

# --- Hierarchical Ward ---
ward = AgglomerativeClustering(linkage='ward', n_clusters=CHOSEN_K)
labels_ward = ward.fit_predict(costumer_preprocessed)

# --- Mean Shift ---
bandwidth = estimate_bandwidth(costumer_preprocessed, quantile=0.2, n_samples=500, random_state=42)
ms = MeanShift(bandwidth=bandwidth, bin_seeding=True, n_jobs=-1)
labels_ms = ms.fit_predict(costumer_preprocessed)
n_clusters_ms = len(set(labels_ms))

print(f"K-Means    : k={CHOSEN_K}")
print(f"Ward       : k={CHOSEN_K}")
print(f"Mean Shift : k={n_clusters_ms} (auto)")

## 2. Quantitative Metrics

Three internal validation metrics (no ground truth needed):

| Metric | Better when | Intuition |
|---|---|---|
| **Silhouette** | **higher** (max 1) | How well each point fits its own cluster vs neighbours |
| **Davies-Bouldin** | **lower** (min 0) | Average similarity between each cluster and its most similar one |
| **Calinski-Harabasz** | **higher** | Ratio of between-cluster to within-cluster dispersion |

In [ ]:
results = {}

for name, labels in [
    ('K-Means',    labels_kmeans),
    ('Ward',       labels_ward),
    ('Mean Shift', labels_ms),
]:
    results[name] = {
        'n_clusters'         : len(set(labels)),
        'silhouette'         : silhouette_score(costumer_preprocessed, labels),
        'davies_bouldin'     : davies_bouldin_score(costumer_preprocessed, labels),
        'calinski_harabasz'  : calinski_harabasz_score(costumer_preprocessed, labels),
    }

metrics_df = pd.DataFrame(results).T.round(4)
metrics_df['n_clusters'] = metrics_df['n_clusters'].astype(int)
print(metrics_df.to_string())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

colors = ['steelblue', 'seagreen', 'coral']
models = metrics_df.index.tolist()

for ax, col, better, title in zip(
    axes,
    ['silhouette', 'davies_bouldin', 'calinski_harabasz'],
    ['higher', 'lower', 'higher'],
    ['Silhouette Score\n(higher = better)',
     'Davies-Bouldin Score\n(lower = better)',
     'Calinski-Harabasz Score\n(higher = better)']
):
    bars = ax.bar(models, metrics_df[col], color=colors, edgecolor='black')
    ax.set_title(title)
    ax.set_ylabel(col)
    # highlight best
    best_idx = metrics_df[col].idxmax() if better == 'higher' else metrics_df[col].idxmin()
    best_pos = models.index(best_idx)
    bars[best_pos].set_edgecolor('gold')
    bars[best_pos].set_linewidth(3)

fig.suptitle(f'Clustering Metrics Comparison (K-Means & Ward: k={CHOSEN_K} | Mean Shift: k={n_clusters_ms})',
             fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## 3. Cluster Size Distribution

Unbalanced clusters are harder to act on from a marketing perspective.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, labels, name, color in zip(
    axes,
    [labels_kmeans, labels_ward, labels_ms],
    ['K-Means', 'Ward', 'Mean Shift'],
    ['steelblue', 'seagreen', 'coral']
):
    sizes = pd.Series(labels).value_counts().sort_index()
    ax.bar(sizes.index.astype(str), sizes.values, color=color, edgecolor='black')
    ax.set_title(f'{name} — cluster sizes')
    ax.set_xlabel('Cluster')
    ax.set_ylabel('Customers')

plt.tight_layout()
plt.show()

## 4. Agreement Between K-Means and Ward

As seen in class, a cross-tab reveals which clusters share the same identity  
across methods. High overlap = robust segmentation.

In [ ]:
crosstab = pd.crosstab(
    pd.Series(labels_kmeans, name='K-Means'),
    pd.Series(labels_ward,   name='Ward')
)
print("K-Means vs Ward cross-tab:")
print(crosstab)

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(crosstab, annot=True, fmt='d', cmap='Blues', ax=ax, linewidths=0.5)
ax.set_title('K-Means vs Ward — customer overlap')
plt.tight_layout()
plt.show()

## 5. Cluster Profile Comparison — Scaled Feature Means

Side-by-side heatmaps to see if all three methods tell the same story.

In [ ]:
tmp = costumer_preprocessed.copy()
tmp['kmeans'] = labels_kmeans
tmp['ward']   = labels_ward
tmp['ms']     = labels_ms

profile_kmeans = tmp.drop(columns=['ward', 'ms']).groupby('kmeans').mean()
profile_ward   = tmp.drop(columns=['kmeans', 'ms']).groupby('ward').mean()
profile_ms     = tmp.drop(columns=['kmeans', 'ward']).groupby('ms').mean()

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

for ax, profile, title in zip(
    axes,
    [profile_kmeans, profile_ward, profile_ms],
    [f'K-Means (k={CHOSEN_K})', f'Ward (k={CHOSEN_K})', f'Mean Shift (k={n_clusters_ms})']
):
    sns.heatmap(
        profile.T,
        cmap='RdBu_r', center=0,
        linewidths=0.3, ax=ax, annot=False,
        cbar=False
    )
    ax.set_title(title)
    ax.set_xlabel('Cluster')

fig.suptitle('Feature means per cluster (scaled) — all three methods', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## 6. Verdict

Run the cells above and fill in the table below based on the results.

In [ ]:
print("=" * 60)
print("SUMMARY")
print("=" * 60)
print(metrics_df.to_string())
print()

best_sil = metrics_df['silhouette'].idxmax()
best_db  = metrics_df['davies_bouldin'].idxmin()
best_ch  = metrics_df['calinski_harabasz'].idxmax()

print(f"Best Silhouette         → {best_sil}")
print(f"Best Davies-Bouldin     → {best_db}")
print(f"Best Calinski-Harabasz  → {best_ch}")
print()

votes = pd.Series([best_sil, best_db, best_ch]).value_counts()
winner = votes.idxmax()
print(f"Recommended algorithm   → {winner} ({votes[winner]}/3 metrics)")
print()
print("Note: also consider cluster balance and interpretability")
print(f"  Mean Shift found k={n_clusters_ms} automatically — check if that is meaningful.")
print(f"  Ward and K-Means both used k={CHOSEN_K} — consistent and comparable.")